# Prototipo de Pipeline: ACLED + GDELT

Este notebook implementa el flujo de trabajo inspirado en la arquitectura de OSINT avanzada para predecir la severidad de conflictos (fatalities).

## 1. Importación de Librerías y Scripts Locales

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Añadir scripts al path
sys.path.append(os.path.abspath('../'))
from scripts.data_processing import process_acled, merge_acled_gdelt

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print("Entorno listo.")

## 2. Carga y Procesamiento de Datos

Asumiendo que ya ejecutaste `scripts/data_extraction.py` o descargaste los CSV manualmente.

In [ ]:
# Rutas de ejemplo (ajusta según tus archivos descargados)
acled_path = RAW_DIR / 'acled_iran.csv'
gdelt_path = RAW_DIR / 'gdelt_ir_daily.csv'

if acled_path.exists() and gdelt_path.exists():
    acled_daily = process_acled(acled_path)
    gdelt_daily = pd.read_csv(gdelt_path)
    
    modeling_df = merge_acled_gdelt(acled_daily, gdelt_daily)
    modeling_df.to_csv(PROCESSED_DIR / 'master_table.csv', index=False)
    print("Tabla maestra creada exitosamente.")
else:
    print("ADVERTENCIA: Archivos de datos no encontrados en data/raw. Ejecuta la extracción primero.")

## 3. Modelo Baseline (HistGradientBoosting)

Utilizamos un modelo de regresión para predecir `fatalities` basándonos en las señales mediáticas de GDELT.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

if 'modeling_df' in locals():
    # Definir features y target
    features = [
        'lat_bin', 'lon_bin', 'gdelt_event_count', 'gdelt_mentions', 
        'gdelt_sources', 'gdelt_articles', 'avg_tone', 'avg_goldstein',
        'material_conflict_events', 'high_conflict_events'
    ]
    target = 'fatalities'
    
    # Ordenar por fecha para validación temporal
    df = modeling_df.sort_values('event_date').dropna(subset=features)
    
    X = df[features]
    y = np.log1p(df[target]) # Transformación logarítmica sugerida
    
    # Split simple temporal (80% train, 20% test)
    split = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]

    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    preds = np.expm1(model.predict(X_test))
    y_test_orig = np.expm1(y_test)
    
    print(f"MAE: {mean_absolute_error(y_test_orig, preds):.4f}")
    print(f"R2 Score: {r2_score(y_test_orig, preds):.4f}")